# Fresh ten-epoch Laya fine-tuning on TMCAD

This independent run starts from the pinned base Laya typed-decisions checkpoint. It uses the **same frozen train/calibration/validation/test population, compact CAD state, Choice/Noul questions, FP16, physical batch 8, accumulation 1, optimizer learning rates, weight decay, and loss** as the four-epoch Kaggle run. The configured maximum changes from 4 to **10**. The original 0.4-to-0.1 RL noise schedule is spread over ten epochs rather than four, so the first four updates are not identical between runs. No previous fine-tuned weights or cached predictions are loaded.

Upload `jev_tmcad_laya_finetune_10ep_v1.zip` as a **private Kaggle Dataset**, attach it under Add Input, select a GPU, and enable Internet for the pinned Laya dependency and base model. Run all cells. The original four-epoch notebook and results remain separate. Test metrics are exploratory because the test split was previously inspected.

Each completed epoch writes `laya_epoch_checkpoints/epoch_NN.safetensors` and `epoch_NN.json`. Validation macro F1 selects the best epoch; its Laya-loadable checkpoint is in `laya_checkpoint/`. Calibration uses the unchanged train-only calibration set. All 9,798 matched states then receive fine-tuned inference, direct Choice evaluation, and source traceability. The final ZIP includes **every epoch checkpoint** and the selected checkpoint. Ten large snapshots plus the ZIP may use roughly 18–20 GB in `/kaggle/working`; ensure adequate free space before starting. The archive is created only after all ten epochs and evaluation succeed.


In [ ]:
from pathlib import Path
import json, sys, zipfile, subprocess

inputs = Path('/kaggle/input')
matches = [p for p in inputs.rglob('manifest.json') if p.parent.name == 'jev_tmcad_laya_ft']
if not matches:
    zips = list(inputs.rglob('jev_tmcad_laya_finetune_10ep_v1.zip'))
    if len(zips) != 1:
        raise FileNotFoundError('Attach exactly one private Dataset containing jev_tmcad_laya_finetune_10ep_v1.zip')
    unpacked = Path('/kaggle/working/unpacked_input')
    unpacked.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as archive:
        if any(Path(n).is_absolute() or '..' in Path(n).parts for n in archive.namelist()):
            raise ValueError('Unsafe archive path')
        archive.extractall(unpacked)
    matches = list(unpacked.rglob('manifest.json'))
if len(matches) != 1:
    raise ValueError(f'Expected one study bundle manifest, found {len(matches)}')
bundle = matches[0].parent
print('Input bundle:', bundle)
print('Installing pinned Laya dependencies; Kaggle CUDA PyTorch is retained')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                       str(bundle / 'config/requirements_kaggle.txt')])


In [ ]:
import hashlib, importlib, platform, shutil, time
import pandas as pd
import torch
import yaml

manifest = json.loads((bundle / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['bundle_type'] == 'jev_tmcad_laya_finetune_10ep_v1' and manifest['training_epochs'] == 10, 'Attach the ten-epoch v1 upload ZIP'
source_file = bundle / 'code/src/jev_tmcad/training/laya.py'
if any(name not in source_file.read_text(encoding='utf-8') for name in ('_validation_macro_f1', '_calibrate_choice_temperature')):
    raise RuntimeError(f'The attached Dataset still has the old trainer: {source_file}. Attach the ten-epoch ZIP.')
for relative, info in manifest['files'].items():
    path = bundle / relative
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != info['sha256']:
        raise ValueError(f'Bundle file missing or changed: {relative}')
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator before training')
print('Verified', len(manifest['files']), 'files and', manifest['n_samples'], 'sample states')
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)
config = yaml.safe_load((bundle / 'config/kaggle_config.yaml').read_text(encoding='utf-8'))
questions = yaml.safe_load((bundle / 'config/questions_compact.yaml').read_text(encoding='utf-8'))
sys.path.insert(0, str(bundle / 'code/src'))
# Rerunning in one Kaggle kernel can retain the old package in Python's cache.
for module_name in list(sys.modules):
    if module_name == 'jev_tmcad' or module_name.startswith('jev_tmcad.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
from jev_tmcad.training.laya import (train_laya_finetune,
    run_finetuned_laya_states, evaluate_finetuned_laya_choice, _validation_macro_f1)
import jev_tmcad.training.laya as trainer_module
if Path(trainer_module.__file__).resolve() != source_file.resolve():
    raise RuntimeError(f'Python imported {trainer_module.__file__}, expected {source_file}')
from jev_tmcad.utils import sha256_json
assert sha256_json(questions) == manifest['question_sha256']
assert f"{config['base_repository']}/{config['base_subfolder']}@{config['base_revision']}" == manifest['base_model']
assert _validation_macro_f1([{'label': 0}, {'label': 1}], [[5, 0], [0, 5]], 2) == 1.0
output = Path('/kaggle/working/jev_tmcad_laya_ft_10epoch_results')
output.mkdir(parents=True, exist_ok=True)
for name in ('kaggle_config.yaml', 'questions_compact.yaml', 'local_study_full_ft.yaml',
             'requirements_kaggle.txt'):
    shutil.copy2(bundle / 'config' / name, output / name)
shutil.copy2(bundle / 'manifest.json', output / 'input_manifest.json')
shutil.copy2(bundle / 'data/metadata.json', output / 'prepared_data_metadata.json')
environment = {'python': platform.python_version(), 'platform': platform.platform(),
               'torch': torch.__version__, 'cuda': torch.version.cuda,
               'gpu': torch.cuda.get_device_name(0), 'config': config}
(output / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')

assert int(config['epochs']) == 10 and config['save_epoch_snapshots'] is True


## Train, validate, and preserve every epoch

Training starts at the pinned base model with a fresh optimizer. The ten epoch weights and scores are saved before the final validation-selected model is calibrated and exported. The progress JSON records the current batch and stage. The test set is never used for selecting an epoch or a temperature. A recovery cell runs first only if all ten epochs have already completed but checkpoint export was interrupted; it never substitutes an older four-epoch run because this notebook uses a separate output directory.


In [ ]:
"""Kaggle notebook cell: finalize a completed Laya run without retraining.

Paste this entire file into a new cell in the *same* Kaggle session after the
notebook setup cell. It also runs automatically in the revised notebook.
"""
import gc
import json
import math
import shutil
from importlib.metadata import version

import laya
import numpy as np
import torch
from safetensors.torch import load_file

from jev_tmcad.data.annotations import file_sha256
from jev_tmcad.training.laya import _base_checkpoint, _logits, _tokenize
from jev_tmcad.utils import read_jsonl, sha256_json, write_json


def recover_completed_laya_training(bundle, output, config, questions):
    """Select the best saved epoch and perform train-only temperature fitting."""
    checkpoint = output / "laya_checkpoint"
    history_path = output / "laya_checkpoint.history.json"
    if (checkpoint / "study_metadata.json").exists():
        return json.loads((checkpoint / "study_metadata.json").read_text(encoding="utf-8"))
    if checkpoint.exists():
        raise FileExistsError(f"Partial checkpoint exists; inspect it before recovery: {checkpoint}")
    history_record = json.loads(history_path.read_text(encoding="utf-8"))
    history = history_record["epochs"]
    expected_epochs = int(config["epochs"])
    if len(history) != expected_epochs or [r["epoch"] for r in history] != list(range(1, expected_epochs + 1)):
        raise ValueError("Training history is incomplete; recovery cannot assume four finished epochs")
    for key in ("batch_size", "accumulation_steps", "precision"):
        if history_record[key] != config[key]:
            raise ValueError(f"Saved training history uses another {key}")
    scores = [float(r["validation_macro_f1"]) for r in history]
    if not all(math.isfinite(s) for s in scores):
        raise ValueError("Validation history contains a nonfinite score")
    best_index = max(range(len(scores)), key=scores.__getitem__)
    best_epoch = best_index + 1
    snapshot = output / "laya_epoch_checkpoints" / f"epoch_{best_epoch:02d}.safetensors"
    summary = json.loads(snapshot.with_suffix(".json").read_text(encoding="utf-8"))
    if (summary["epoch"] != best_epoch or
            float(summary["validation_macro_f1"]) != scores[best_index] or
            summary["model_sha256"] != file_sha256(snapshot)):
        raise ValueError("Selected epoch snapshot differs from the validation history")
    print(f"Using saved epoch {best_epoch}/{expected_epochs} (validation macro F1 {scores[best_index]:.4f}); no training batches will run.", flush=True)

    data_dir = bundle / "data"
    data_meta = json.loads((data_dir / "metadata.json").read_text(encoding="utf-8"))
    if data_meta["question_sha256"] != sha256_json(questions):
        raise ValueError("Choice question differs from the frozen fine-tuning data")
    options = list(questions["classification"]["criteria"])
    if data_meta["options"] != options:
        raise ValueError("Choice option order differs from the frozen fine-tuning data")
    data_hashes = {name: file_sha256(data_dir / f"{name}.jsonl") for name in
                   ("train", "calibration", "validation")}
    if data_hashes != data_meta["partition_hashes"]:
        raise ValueError("Frozen training, calibration, or validation data changed")
    cal_rows = read_jsonl(data_dir / "calibration.jsonl")
    if not cal_rows or any(r["split"] != "train" or r["partition"] != "calibration" for r in cal_rows):
        raise ValueError("Calibration must contain train-only samples")

    # The failed trainer may still be retained by IPython's exception traceback.
    import sys
    for name in ("last_traceback", "last_value", "last_type"):
        if hasattr(sys, name):
            setattr(sys, name, None)
    try:
        shell = get_ipython()
    except NameError:
        shell = None
    if shell is not None and getattr(shell, "last_execution_result", None) is not None:
        shell.last_execution_result.error_in_exec = None
    gc.collect()
    torch.cuda.empty_cache()
    print("Loading pinned base Laya and fitting temperature on train-only calibration records...", flush=True)
    base, resolved = _base_checkpoint(config["base_repository"],
                                      config["base_subfolder"], config["base_revision"])
    agent = laya.load(str(base), device=config["device"])
    cal = _tokenize(cal_rows, agent.tok, agent.cfg, questions["classification"])
    agent.model.load_state_dict(load_file(str(snapshot)), strict=True)
    z = np.asarray(_logits(agent.model, cal, agent.tok.pad_token_id,
                           int(config["batch_size"]), agent.device, config["precision"]), dtype=float)
    y = np.asarray([int(item["label"]) for item in cal], dtype=int)
    if z.shape != (len(y), len(options)) or not np.isfinite(z).all() or np.any((y < 0) | (y >= len(options))):
        raise ValueError("Calibration logits or Choice indices are invalid")

    def nll(temp):
        scaled = z / temp
        scaled -= scaled.max(axis=1, keepdims=True)
        logp = scaled - np.log(np.exp(scaled).sum(axis=1, keepdims=True))
        return -float(np.mean(logp[np.arange(len(y)), y]))

    temperature = float(min(np.geomspace(.5, 5., 81), key=nll))
    checkpoint.mkdir(parents=True)
    agent.model.encoder.save_pretrained(checkpoint / "encoder", safe_serialization=True)
    agent.tok.save_pretrained(checkpoint / "tokenizer")
    shutil.copy2(snapshot, checkpoint / "model.safetensors")
    model_cfg = dict(agent.cfg)
    model_cfg.pop("temperature_by_options", None)
    model_cfg["temperature"] = [temperature, 1.0, 1.0]
    write_json(checkpoint / "rl_agent_config.json", model_cfg)
    progress_path = output / "laya_checkpoint.progress.json"
    progress = json.loads(progress_path.read_text(encoding="utf-8")) if progress_path.exists() else {}
    meta = {"base_repository": config["base_repository"], "base_subfolder": config["base_subfolder"],
            "base_revision": resolved, "laya_version": version("laya"),
            "data_metadata": str(data_dir / "metadata.json"), "data_hashes": data_hashes,
            "best_validation_macro_f1": scores[best_index], "best_epoch": best_epoch,
            "calibration_temperature": temperature, "calibration_nll": nll(temperature),
            "training_seconds": None, "elapsed_seconds_at_failure": progress.get("elapsed_seconds"),
            "accumulation_steps": int(config["accumulation_steps"]),
            "batch_size": int(config["batch_size"]), "precision": config["precision"],
            "save_epoch_snapshots": True, "epochs": history,
            "checkpoint_sha256": file_sha256(checkpoint / "model.safetensors"),
            "recovered_from_snapshot": str(snapshot),
            "recovery_note": "All training epochs completed before calibration failed; no optimizer steps were rerun."}
    write_json(checkpoint / "study_metadata.json", meta)
    del agent
    gc.collect()
    torch.cuda.empty_cache()
    return meta


if (output / "laya_checkpoint.history.json").exists() and not (output / "laya_checkpoint" / "study_metadata.json").exists():
    training = recover_completed_laya_training(bundle, output, config, questions)
    print("Recovered without retraining. Best validation epoch:", training["best_epoch"],
          "| calibration temperature:", training["calibration_temperature"])


In [ ]:
from importlib.metadata import version
environment['laya'] = version('laya')
(output / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
checkpoint = output / 'laya_checkpoint'
if not (checkpoint / 'study_metadata.json').exists():
    training = train_laya_finetune(
        bundle / 'data', checkpoint, questions,
        repository=config['base_repository'], subfolder=config['base_subfolder'],
        revision=config['base_revision'], seed=int(config['project_seed']),
        device=config['device'], epochs=int(config['epochs']),
        batch_size=int(config['batch_size']),
        accumulation_steps=int(config['accumulation_steps']),
        precision=config['precision'],
        save_epoch_snapshots=bool(config['save_epoch_snapshots']),
        encoder_lr=float(config['encoder_lr']), head_lr=float(config['head_lr']),
        weight_decay=float(config['weight_decay']), group_size=int(config['group_size']),
        sigma_start=float(config['sigma_start']), sigma_end=float(config['sigma_end']),
        rl_weight=float(config['rl_weight']))
else:
    training = json.loads((checkpoint / 'study_metadata.json').read_text(encoding='utf-8'))
pd.DataFrame(training['epochs']).to_csv(output / 'training_history.csv', index=False)
print('Best validation macro F1:', training['best_validation_macro_f1'],
      'at epoch', training['best_epoch'])


In [ ]:
from jev_tmcad.data.annotations import file_sha256
if len(training['epochs']) != 10 or [row['epoch'] for row in training['epochs']] != list(range(1, 11)):
    raise ValueError('Ten completed epoch records are required before inference')
snapshot_dir = output / 'laya_epoch_checkpoints'
for row in training['epochs']:
    snapshot = snapshot_dir / f"epoch_{row['epoch']:02d}.safetensors"
    summary = json.loads(snapshot.with_suffix('.json').read_text(encoding='utf-8'))
    if (summary['epoch'] != row['epoch'] or
            summary['validation_macro_f1'] != row['validation_macro_f1'] or
            summary['model_sha256'] != file_sha256(snapshot)):
        raise ValueError(f"Saved epoch {row['epoch']} is missing or differs from validation history")
expected_best = 1 + max(range(10), key=lambda i: training['epochs'][i]['validation_macro_f1'])
if training['best_epoch'] != expected_best or training['checkpoint_sha256'] != file_sha256(checkpoint / 'model.safetensors'):
    raise ValueError('Selected best checkpoint differs from validated training history')
print('Verified all ten epoch models and best selected epoch:', expected_best)


## Run the selected model on the matched cohort

The following stage uses the frozen per-sample compact states and the unchanged Choice/Noul question specification. It writes the JSONL record format expected by local `study-evaluate`. Existing complete records are reused by request hash, and the file is appended incrementally.


In [ ]:
states = bundle / 'data/inference_states.jsonl'
results = output / 'laya_results.jsonl'
if config['run_full_inference']:
    n_records = run_finetuned_laya_states(states, questions, checkpoint, results,
                                           device=config['device'])
    if n_records != manifest['n_samples']:
        raise ValueError(f'Inference coverage: {n_records} of {manifest["n_samples"]}')
    print('Fine-tuned Laya responses:', n_records)


## Direct Choice report and downloadable output

These are direct Choice metrics only. The local evaluator remains authoritative for the full method matrix and expert-graded retrieval. Test metrics are descriptive because this test split had already been inspected before fine-tuning was added.


In [ ]:
if config['run_full_inference'] and config['run_direct_choice_evaluation']:
    direct = evaluate_finetuned_laya_choice(states, results, output / 'direct_choice_evaluation')
    print(pd.DataFrame(direct['methods']).T[['n', 'accuracy', 'macro_f1', 'balanced_accuracy', 'nll', 'brier']])

readme = '''Fresh ten-epoch Kaggle Laya/TMCAD output\n\n'''
readme += 'laya_checkpoint/: Laya-loadable selected model (model.safetensors, encoder/, tokenizer/, rl_agent_config.json, study_metadata.json).\n'
readme += 'laya_epoch_checkpoints/: epoch weight snapshots and validation summaries.\n'
readme += 'laya_checkpoint.history.json, laya_checkpoint.progress.json, training_history.csv: training logs; all ten epochs are preserved.\n'
readme += 'laya_results.jsonl: matched all-sample Choice/Noul predictions, request hashes, source metadata and timing.\n'
readme += 'direct_choice_evaluation/: per-sample predictions, split metrics and confusion tables.\n'
readme += 'config/*.yaml and input_manifest.json: frozen settings and source file hashes.\n\n'
readme += 'Locally, verify/import this ZIP with kaggle/laya/import_results.py, then use configs/study_full_ft_kaggle.yaml for the matched evaluator. Retrieval also requires finalized expert grades.\n'
(output / 'README_outputs.txt').write_text(readme, encoding='utf-8')
files = [p for p in output.rglob('*') if p.is_file() and p.name != 'output_manifest.json']
result_manifest = {'input_manifest_sha256': hashlib.sha256((bundle / 'manifest.json').read_bytes()).hexdigest(),
                   'model_sha256': hashlib.sha256((checkpoint / 'model.safetensors').read_bytes()).hexdigest(),
                   'files': {str(p.relative_to(output)): {'sha256': hashlib.sha256(p.read_bytes()).hexdigest(),
                                                           'bytes': p.stat().st_size} for p in sorted(files)}}
(output / 'output_manifest.json').write_text(json.dumps(result_manifest, indent=2), encoding='utf-8')
archive_path = Path('/kaggle/working/jev_tmcad_laya_ft_10epoch_results.zip')
# A second copy of all large epoch weights is needed while building the ZIP.
# Never let archive creation exhaust Kaggle storage and jeopardize saved models.
archive_bytes_upper_bound = sum(path.stat().st_size for path in output.rglob('*') if path.is_file())
free_bytes = shutil.disk_usage('/kaggle/working').free
if free_bytes < archive_bytes_upper_bound + 1024**3:
    print('Insufficient free space for a safe all-files ZIP; models and reports remain in:', output)
    print('Free bytes:', free_bytes, '| estimated uncompressed archive bytes:', archive_bytes_upper_bound)
else:
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as archive:
        for path in sorted(output.rglob('*')):
            if path.is_file():
                archive.write(path, arcname=str(Path(output.name) / path.relative_to(output)))
    print('Download:', archive_path)
print('Output folder:', output)

print('Selected best epoch:', training['best_epoch'], 'of 10')
